# Videos de la presentación · semana del 22

Los ocho videos de los experimentos, ordenados como en la presentación. Cada experimento muestra a la **política normal** (el actor junta los datos) al lado de la **política random** (el azar junta los datos).

| Experimento | Vista del agente |
|---|---|
| 1 | Mapa completo |
| 2 | Ventana 7×7 |
| 3 | Ventana 5×5 |
| 4 | Ventana 3×3 |

En todos los videos juega el **actor aprendido** del último checkpoint, eligiendo siempre su acción más probable, igual que en las evaluaciones. Es la partida de semilla 0. El encabezado del video indica los pasos entrenados.

**Usa CPU y ejecuta las secciones 1 a 3.** Si algún video aparece como anterior al último checkpoint, la sección 4 lo vuelve a crear sin tocar los originales.

## 1. Montar Drive

Usa la cuenta donde están los entrenamientos.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Los ocho runs

Las carpetas de Drive de cada experimento. Los videos nuevos de la sección 4 se guardan en `galleta/presentacion/videos/`.

In [ ]:
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta")
PRESENTACION = RAIZ / "presentacion"
SEMILLA_VIDEO = 0

EXPERIMENTOS = [
    ("Experimento 1", "Mapa completo", {
        "normal": RAIZ / "semana8/cookie_full_hall5_s0_base",
        "random": RAIZ / "semana22/cookie_full_hall5_s0_random",
    }),
    ("Experimento 2", "Ventana 7×7", {
        "normal": RAIZ / "semana15/cookie_partial_hall5_view7_s0_base",
        "random": RAIZ / "semana22/cookie_partial_hall5_view7_s0_random",
    }),
    ("Experimento 3", "Ventana 5×5", {
        "normal": RAIZ / "semana15/cookie_partial_hall5_view5_s0_base",
        "random": RAIZ / "semana22/cookie_partial_hall5_view5_s0_random",
    }),
    ("Experimento 4", "Ventana 3×3", {
        "normal": RAIZ / "semana15/cookie_partial_hall5_view3_s0_base",
        "random": RAIZ / "semana22/cookie_partial_hall5_view3_s0_random",
    }),
]
POLITICAS = {
    "normal": "Política normal (recolecta el actor)",
    "random": "Política random (recolecta el azar)",
}

for experimento, vista, runs in EXPERIMENTOS:
    for politica, ruta in runs.items():
        estado = "ok" if (ruta / "latest.pt").exists() else "NO ENCONTRADO"
        print(f"{experimento} · {politica:6} · {ruta.name}: {estado}")

## 3. Ver los videos

Para cada run se muestra el video más reciente de la semilla 0, ya sea el original del run o uno creado en la sección 4. Debajo de cada video se indica si se hizo con el último checkpoint.

In [ ]:
import html
from IPython.display import HTML, Video, display


def video_de(ruta):
    """El video más reciente de la semilla: el del run o el de la presentación."""
    candidatos = []
    for carpeta in (ruta, PRESENTACION / "videos" / ruta.name):
        archivo = carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4"
        try:
            if archivo.is_file() and archivo.stat().st_size > 0:
                candidatos.append((archivo.stat().st_mtime, archivo))
        except OSError:
            pass  # Drive puede estar sincronizando el archivo
    return max(candidatos)[1] if candidatos else None


def al_dia(ruta, archivo):
    """El video está al día si se creó después del último checkpoint del run."""
    checkpoint = ruta / "latest.pt"
    return checkpoint.exists() and archivo.stat().st_mtime >= checkpoint.stat().st_mtime


def tarjeta(politica, ruta):
    titulo = f"<h4>{html.escape(POLITICAS[politica])}</h4>"
    archivo = video_de(ruta)
    if archivo is None:
        return f'<div class="tarjeta">{titulo}<p>No hay video. Créalo en la sección 4.</p></div>'
    try:
        reproductor = Video(str(archivo), embed=True)._repr_html_()
    except OSError:
        return f'<div class="tarjeta">{titulo}<p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p></div>'
    if al_dia(ruta, archivo):
        nota = "Hecho con el último checkpoint."
    else:
        nota = "⚠️ Anterior al último checkpoint: créalo de nuevo en la sección 4."
    return (f'<div class="tarjeta">{titulo}{reproductor}'
            f'<p><small>{html.escape(nota)}<br>{html.escape(str(archivo.relative_to(RAIZ)))}</small></p></div>')


def mostrar_videos():
    estilos = (
        "<style>"
        ".experimento{max-width:1100px;margin:0 0 28px}"
        ".experimento h3{margin:0 0 10px}"
        ".pareja{display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:16px}"
        ".tarjeta{border:1px solid #8885;border-radius:8px;padding:12px}"
        ".tarjeta h4{margin:0 0 8px}"
        ".tarjeta video{width:100%;height:auto}"
        "</style>"
    )
    bloques = []
    for experimento, vista, runs in EXPERIMENTOS:
        tarjetas = "".join(tarjeta(politica, ruta) for politica, ruta in runs.items())
        bloques.append(f'<div class="experimento"><h3>{html.escape(experimento)} · {html.escape(vista)}</h3>'
                       f'<div class="pareja">{tarjetas}</div></div>')
    display(HTML(estilos + "".join(bloques)))


mostrar_videos()

## 4. Crear de nuevo los videos desactualizados (opcional, en CPU)

Cambia `CREAR_VIDEOS` a `True` y ejecuta la celda. Crea un video con el último checkpoint para cada run cuyo video falte o sea anterior a ese checkpoint. La primera vez descarga el repositorio e instala las dependencias en un entorno aparte, con PyTorch para CPU.

Solo se copian el checkpoint y su configuración a una carpeta temporal. Los videos nuevos quedan en `galleta/presentacion/videos/` y los originales no se tocan. Puede tardar varios minutos por video. Al terminar se actualiza la galería.

In [ ]:
CREAR_VIDEOS = False

if CREAR_VIDEOS:
    import os
    import shutil
    import subprocess
    import sys
    import tempfile

    REPO = Path("/content/r2dreamer-cookie")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/FabriRandon/r2dreamer-cookie.git", str(REPO)], check=True)

    pendientes = []
    for experimento, vista, runs in EXPERIMENTOS:
        for politica, ruta in runs.items():
            archivo = video_de(ruta)
            if archivo is not None and al_dia(ruta, archivo):
                continue
            if not (ruta / "latest.pt").exists() or not (ruta / ".hydra/config.yaml").exists():
                print(f"{experimento} · {politica}: no hay checkpoint en Drive.")
                continue
            pendientes.append((f"{experimento} · {politica}", ruta))

    if pendientes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv")
        if uv is None:
            raise RuntimeError("No se encontró uv después de instalarlo. Vuelve a ejecutar la celda.")
        entorno = REPO / ".venv-videos"
        python_video = entorno / "bin/python"
        if not python_video.exists():
            subprocess.run([uv, "venv", "--python", "3.11", str(entorno)], check=True)
        with tempfile.TemporaryDirectory(prefix="video-dependencias-") as temporal:
            restricciones = Path(temporal) / "cpu.txt"
            restricciones.write_text("torch==2.8.0+cpu\n")
            subprocess.run([uv, "pip", "install", "--python", str(python_video),
                            "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                            "--index-strategy", "unsafe-best-match", "--constraint", str(restricciones),
                            "-e", ".[cookie]"], cwd=REPO, check=True)
        for nombre, ruta in pendientes:
            destino = PRESENTACION / "videos" / ruta.name
            destino.mkdir(parents=True, exist_ok=True)
            video = destino / f"agente_episodio{SEMILLA_VIDEO}.mp4"
            provisional = destino / f"agente_episodio{SEMILLA_VIDEO}.tmp.mp4"
            print(f"Creando video: {nombre}")
            try:
                with tempfile.TemporaryDirectory(prefix="modelo-para-video-") as temporal:
                    copia = Path(temporal)
                    (copia / ".hydra").mkdir()
                    shutil.copyfile(ruta / ".hydra/config.yaml", copia / ".hydra/config.yaml")
                    shutil.copyfile(ruta / "latest.pt", copia / "latest.pt")
                    proceso = subprocess.run([str(python_video), "watch.py", "--logdir", str(copia),
                                              "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                             cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                             text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
                if proceso.returncode == 0 and provisional.exists() and provisional.stat().st_size > 0:
                    provisional.replace(video)
                    print(proceso.stdout.strip().splitlines()[-1].split("Video en ")[0])
                    print("Guardado:", video)
                else:
                    print(f"No se pudo crear {nombre}; puedes reintentar esta celda.")
                    print("\n".join(proceso.stdout.splitlines()[-10:]))
            except OSError as error:
                print(f"No se pudo leer el checkpoint de {nombre}: {error}. Reintenta cuando Drive se sincronice.")
    else:
        print("Todos los videos están hechos con el último checkpoint.")
    mostrar_videos()
else:
    print("Para crear los videos desactualizados, cambia CREAR_VIDEOS a True.")